# Supplementary Figure 5 | lncRNA expression level and tissue specificity

**a.** Distribution of each gene’s median TPM across samples within each tissue, one panel per biotype (novel lncRNA, existing lncRNA, protein-coding, and pseudogene). The vertical dotted line marks the median of per-tissue medians. Novel lncRNAs are expressed at lower levels than other biotypes.

**b.** Cumulative τ distributions stratified by max-expression quintile (Q1-Q5) for novel and existing lncRNAs, protein-coding genes, and pseudogenes. Shaded bands mark housekeeping (τ ≤ 0.15), midrange (0.15 < τ ≤ 0.85), and tissue-specific (τ > 0.85) ranges. Within each expression bin, novel lncRNAs remain more tissue-specific than existing lncRNAs and protein-coding genes, with the strongest specificity in the lowest expression quintiles.

**c.** Odds ratios for tissue specificity (T.S.: τ > 0.85) by lncRNA genomic context for novel and existing lncRNAs, adjusted for expression (expr: log10(max TPM + 1)). Intergenic lncRNAs are the reference (diamond), and are most likely to be tissue-specific for both novel and existing lncRNAs. Points, logistic-regression odds ratios (ORs); error bars, 95% CIs; asterisks, two-sided Wald p (* p < 0.05; ** p < 0.01; *** p < 0.001).

**d.** Counts of tissue-specific (T.S.: τ > 0.85) genes by highest median-expressed organ for novel lncRNA, existing lncRNA, protein-coding, and pseudogene. For each organ and biotype, open and filled points show counts before vs after testis rescue: genes whose highest-median expressed organ is testis but that remain tissue-specific when testis is excluded are reassigned to their highest median expressed non-testis organ.


## Setup


In [ ]:
import gzip
import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import statsmodels.formula.api as smf
from matplotlib.lines import Line2D
from statsmodels.stats.multitest import multipletests

plt.rc("axes", labelsize=6)
plt.rc("xtick", labelsize=6)
plt.rc("ytick", labelsize=6)
plt.rc("legend", fontsize=5)
plt.rc("legend", title_fontsize=6)
plt.rc("font", size=6)
plt.rc("figure", titlesize=7)
plt.rcParams["pdf.fonttype"] = 42
plt.rcParams["ps.fonttype"] = 42
plt.rcParams["figure.figsize"] = (7.2, 5.0)
plt.rcParams["figure.dpi"] = 300
plt.rcParams["savefig.dpi"] = 300

fig_width, fig_height, fig_dpi = 7.2, 5.0, 300
fdr_alpha = 0.05
expected_n_bulk_tissues = 54  # histology subtypes excluded; includes 2 cell lines
expected_n_tau_tissues = 52   # bulk set minus cell lines
tau_housekeeping = 0.15
tau_tissue_specific = 0.85
category_edges = [0, tau_housekeeping, tau_tissue_specific, 1]
category_labels = ["Housekeeping", "Midrange", "Tissue specific"]

# Histology subtypes dropped from the official median TPM GCT (→ 54 bulk sites); cell lines also dropped for τ (→ 52)
histology_subtypes = {
    "Colon - Transverse - Muscularis",
    "Stomach - Mixed Cell",
    "Pancreas - Mixed Cell",
    "Small Intestine - Terminal Ileum - Mixed Cell",
    "Stomach - Mucosa",
    "Stomach - Muscularis",
    "Pancreas - Acini",
    "Small Intestine - Terminal Ileum - Lymphoid Aggregate",
    "Colon - Transverse - Mixed Cell",
    "Colon - Transverse - Mucosa",
    "Liver - Mixed Cell",
    "Pancreas - Islets",
    "Liver - Hepatocyte",
    "Liver - Portal Tract",
}
cell_line_tissues = {
    "Cells - EBV-transformed lymphocytes",
    "Cells - Cultured fibroblasts",
}

gene_type_pretty = {
    "novel_lncRNA": "novel lncRNA",
    "existing_lncRNA": "existing lncRNA",
    "protein_coding": "protein-coding",
    "pseudogene": "pseudogene",
}
gene_type_palette = {
    "novel_lncRNA": sns.color_palette("Paired", 12)[5],
    "existing_lncRNA": sns.color_palette("Paired", 12)[4],
    "protein_coding": sns.color_palette("Paired", 12)[1],
    "pseudogene": sns.color_palette("Paired", 12)[3],
}
organ_bar_biotypes = ["novel_lncRNA", "existing_lncRNA", "protein_coding", "pseudogene"]
expr_bin_labels = ["Q1(lowest)", "Q2", "Q3", "Q4", "Q5(highest)"]
gene_types = ["novel_lncRNA", "existing_lncRNA", "protein_coding", "pseudogene", "sRNA"]
odds_gene_types = ["novel_lncRNA", "existing_lncRNA"]
reference_cat = "intergenic"
lnc_cat_order = [
    "intergenic", "bidirectional", "antisense_overlapping", "antisense_intronic",
    "sense_overlapping", "sense_intronic",
]
lnc_cat_pretty = {
    "intergenic": "Intergenic",
    "bidirectional": "Bidirectional",
    "antisense_overlapping": "Antisense\noverlapping",
    "antisense_intronic": "Antisense\nintronic",
    "sense_overlapping": "Sense\noverlapping",
    "sense_intronic": "Sense\nintronic",
}

## Paths

In [ ]:
gtex_colors_json = Path("refs/colors.json")
tissue_names_file = Path("annotation/refs/tissue_name.txt")
gtex_sample_meta = Path("path/to/GTEx_Analysis_v10_Annotations_SampleAttributesDS.txt")
gencode_class = Path("path/to/gencode_v39_v47_gene_classification.txt")
gtex_median_tpm = Path("path/to/GTEx_Analysis_2025-08-22_v11_RNASeQCv2.4.3_gene_median_tpm.gct.gz")
lnc_category_file = Path("path/to/lncrna_category.txt")
gtex_pooled_null = Path("path/to/GTEx.null_quantifications.txt")

fig_dir = Path("figures")
fig_pdf = fig_dir / "supp_figure_5_expression_tau.pdf"
fig_png = fig_dir / "supp_figure_5_expression_tau.png"


## Load data


### Tissue metadata


In [ ]:
with open(gtex_colors_json) as f:
    _colors = json.load(f)
    tissue_df = pd.DataFrame.from_dict(_colors, orient="index")
tissue_df = tissue_df.reset_index(names="tissue_name")
tissue_df["tissue_color_hex"] = "#" + tissue_df["tissue_color_hex"]
tissue_df["tissue_file_name"] = (
    tissue_df["tissue_name"]
    .str.replace("(", "", regex=False)
    .str.replace(")", "", regex=False)
    .str.replace(" - ", "_")
    .str.replace(" ", "_")
)
tissue_palette = dict(zip(tissue_df["tissue_name"], tissue_df["tissue_color_hex"]))

sample_meta = pd.read_csv(gtex_sample_meta, sep="\t", low_memory=False)
print(f"GTEx tissue colors: {len(tissue_df):,}")


### GENCODE classification


In [ ]:
gencode_ids = pd.read_table(gencode_class, low_memory=False)
print(f"GENCODE genes: {len(gencode_ids):,}")


### Tissue median TPM (official GTEx v11 release)


In [ ]:
with gzip.open(gtex_median_tpm, "rt") as f:
    f.readline()
    f.readline()
    raw = pd.read_csv(f, sep="\t")
median_tpm = raw.set_index("Name").drop(columns=["Description"], errors="ignore")

tissue_map = pd.read_csv(tissue_names_file, sep="\t")
stem_to_pretty = dict(zip(tissue_map["tissue_stem"], tissue_map["tissue_name"]))
renamed = {}
for c in median_tpm.columns:
    pretty = stem_to_pretty.get(c) or stem_to_pretty.get(c.replace("-", "_"))
    if pretty is None:
        raise KeyError(f"Cannot map official median tissue column {c!r} to SMTSD")
    renamed[c] = pretty
median_tpm = median_tpm.rename(columns=renamed)

_before = median_tpm.shape[1]
median_tpm_bulk = median_tpm[[c for c in median_tpm.columns if c not in histology_subtypes]]
if median_tpm_bulk.shape[1] != expected_n_bulk_tissues:
    raise ValueError(
        f"Expected {expected_n_bulk_tissues} bulk tissues after dropping histology subtypes, "
        f"found {median_tpm_bulk.shape[1]}"
    )

# τ / peak-organ panels: 54 bulk minus 2 cell lines → 52
median_tpm = median_tpm_bulk[[c for c in median_tpm_bulk.columns if c not in cell_line_tissues]]
print(
    f"Official median TPM: {_before} GCT columns → {median_tpm_bulk.shape[1]} bulk "
    f"(subtypes out) → {median_tpm.shape[1]} τ (cell lines out)"
)
if median_tpm.shape[1] != expected_n_tau_tissues:
    raise ValueError(
        f"Expected {expected_n_tau_tissues} τ tissues, found {median_tpm.shape[1]}"
    )

loaded_tissues = list(median_tpm.columns)
median_expression_df = median_tpm.reset_index()
print(f"Median TPM for {len(loaded_tissues)} τ tissues; genes: {len(median_expression_df):,}")

### lncRNA category labels


In [ ]:
lnc_cat = pd.read_table(
    lnc_category_file,
    usecols=[
        "gene_id", "genomic_context", "is_tissue_specific",
        "rescued_from_testis", "expressed_gtex",
    ],
)
print(f"lncRNA labels: {len(lnc_cat):,}")
print(
    f"  expressed_gtex={int(lnc_cat['expressed_gtex'].sum()):,}; "
    f"is_tissue_specific={int(lnc_cat['is_tissue_specific'].sum()):,}; "
    f"rescued_from_testis={int(lnc_cat['rescued_from_testis'].sum()):,}"
)


### GTEx null quantifications


In [ ]:
gtex_null = pd.read_table(gtex_pooled_null)
tested_mask = gtex_null["GTEx.mean_counts"] > 0
pvals = gtex_null.loc[tested_mask, "GTEx.null_p"]
rej, _, _, _ = multipletests(pvals, alpha=fdr_alpha, method="fdr_bh")
gtex_null["GTEx.fdr05_quantified"] = False
gtex_null.loc[tested_mask, "GTEx.fdr05_quantified"] = rej
expressed_gene_ids = set(gtex_null.loc[gtex_null["GTEx.fdr05_quantified"], "gene_id"])
print(f"FDR5-expressed genes: {len(expressed_gene_ids):,}")

## Run analysis


### τ from median TPM


In [ ]:
mat = median_expression_df.set_index(median_expression_df.columns[0])
log_tpm = np.log10(mat + 1)
max_expr = log_tpm.max(axis=1)
normed = log_tpm.div(max_expr, axis=0)
n_tissues = log_tpm.shape[1]
tau = (1 - normed).sum(axis=1) / (n_tissues - 1)
tau[max_expr == 0] = np.nan
tau_df = tau.rename("tau").reset_index()
tau_df = tau_df.rename(columns={tau_df.columns[0]: "Name"})
tau_df = tau_df.merge(
    gencode_ids[["gene_id_v47", "gene_type_grouped"]],
    left_on="Name",
    right_on="gene_id_v47",
    how="inner",
)
print(f"tau genes: {len(tau_df):,}")


### Expression melt for TPM KDE


In [ ]:
expression_merged = pd.merge(
    gencode_ids,
    median_expression_df,
    left_on="gene_id_v47",
    right_on="Name",
    how="left",
)
expression_cols = loaded_tissues
expression_merged[expression_cols] = expression_merged[expression_cols].fillna(0)

expression_plot_df = expression_merged.melt(
    id_vars=["gene_id_stable", "gene_type_grouped"],
    value_vars=expression_cols,
    var_name="Tissue",
    value_name="Expression",
)
wanted_types = ["novel_lncRNA", "existing_lncRNA", "protein_coding", "pseudogene"]
plot_df = expression_plot_df[expression_plot_df["gene_type_grouped"].isin(wanted_types)].copy()
plot_df = plot_df[plot_df["Expression"] > 0]
print(f"Expression plot rows (TPM>0): {len(plot_df):,}")


### τ stratified by expression quintile


In [ ]:
tau_expr = tau_df.copy()
tau_expr["max_tpm"] = median_tpm.max(axis=1).reindex(tau_expr["Name"]).values
tau_strat = tau_expr[
    tau_expr["gene_type_grouped"].isin(gene_types) & (tau_expr["max_tpm"] > 0)
].dropna(subset=["tau"]).copy()
tau_strat["log_max_tpm"] = np.log10(tau_strat["max_tpm"] + 1)
tau_strat["expr_bin"], expr_bin_edges_log = pd.qcut(
    tau_strat["log_max_tpm"], q=5, labels=expr_bin_labels, retbins=True
)
expr_bin_edges_tpm = np.power(10.0, expr_bin_edges_log) - 1.0


def _fmt_tpm(x):
    ax = abs(x)
    if ax >= 1000:
        return f"{x:.1e}"
    if ax >= 10:
        return f"{x:.0f}"
    if ax >= 1:
        return f"{x:.1f}"
    if ax >= 0.1:
        return f"{x:.2f}"
    return f"{x:.3f}"


expr_bin_titles = {}
for i, lab in enumerate(expr_bin_labels):
    lo, hi = expr_bin_edges_tpm[i], expr_bin_edges_tpm[i + 1]
    if i == 0:
        expr_bin_titles[lab] = f"Q1: < {_fmt_tpm(hi)} TPM"
    elif i == len(expr_bin_labels) - 1:
        expr_bin_titles[lab] = f"Q5: TPM > {_fmt_tpm(lo)}"
    else:
        expr_bin_titles[lab] = f"Q{i + 1}: ({_fmt_tpm(lo)}-{_fmt_tpm(hi)}) TPM"
print(f"tau_strat genes: {len(tau_strat):,}")
print(tau_strat["expr_bin"].value_counts().sort_index().to_string())
for lab in expr_bin_labels:
    print(f"  {expr_bin_titles[lab]}")


### Tissue-specific ORs by genomic context


In [ ]:
def sig_stars(p):
    if not np.isfinite(p):
        return ""
    if p < 0.001:
        return "***"
    if p < 0.01:
        return "**"
    if p < 0.05:
        return "*"
    return ""


lncrna_tau = tau_strat[tau_strat["gene_type_grouped"].isin(odds_gene_types)].copy()
lncrna_tau = lncrna_tau.merge(lnc_cat, left_on="gene_id_v47", right_on="gene_id", how="inner")
lncrna_tau = lncrna_tau.loc[lncrna_tau["expressed_gtex"]].dropna(subset=["tau"]).copy()
lncrna_tau["tissue_specific"] = lncrna_tau["is_tissue_specific"].astype(int)
lncrna_tau["genomic_context"] = pd.Categorical(
    lncrna_tau["genomic_context"], categories=lnc_cat_order, ordered=True
)
mod_data = lncrna_tau.loc[lncrna_tau["genomic_context"].notnull()].copy()

odds_rows = []
for gt in odds_gene_types:
    gt_data = mod_data[mod_data["gene_type_grouped"] == gt].copy()
    _logit = smf.logit(
        f"tissue_specific ~ log_max_tpm + C(genomic_context, Treatment(reference='{reference_cat}'))",
        data=gt_data,
    ).fit(disp=False)
    for cat in lnc_cat_order:
        if cat == reference_cat:
            odds_rows.append({
                "gene_type_grouped": gt,
                "genomic_context": cat,
                "OR": 1.0,
                "CI_low": 1.0,
                "CI_high": 1.0,
                "pvalue": np.nan,
            })
        else:
            param_name = f"C(genomic_context, Treatment(reference='{reference_cat}'))[T.{cat}]"
            coef = _logit.params.get(param_name, np.nan)
            ci = _logit.conf_int().loc[param_name] if param_name in _logit.conf_int().index else [np.nan, np.nan]
            pvalue = _logit.pvalues.get(param_name, np.nan)
            odds_rows.append({
                "gene_type_grouped": gt,
                "genomic_context": cat,
                "OR": np.exp(coef),
                "CI_low": np.exp(ci[0]),
                "CI_high": np.exp(ci[1]),
                "pvalue": pvalue,
            })

odds_df = pd.DataFrame(odds_rows)
print(
    f"lncRNA genomic-context OR models: {len(mod_data):,} FDR5 genes "
    f"({int(mod_data['tissue_specific'].sum()):,} tissue-specific)"
)
print(odds_df.round(3).to_string(index=False))


### Hybrid peak-organ assignment


In [ ]:
def get_tau_category(tau_val):
    for i in range(len(category_edges) - 1):
        if category_edges[i] <= tau_val < category_edges[i + 1] or (
            i == len(category_edges) - 2 and tau_val == category_edges[-1]
        ):
            return category_labels[i]
    return np.nan


def compute_tau(tpm_df):
    log_tpm_ = np.log10(tpm_df + 1)
    max_expr_ = log_tpm_.max(axis=1)
    normed_ = log_tpm_.div(max_expr_, axis=0)
    n_t = log_tpm_.shape[1]
    tau_ = (1 - normed_).sum(axis=1) / (n_t - 1)
    tau_[max_expr_ == 0] = np.nan
    tau_.name = "tau"
    return tau_


median_tpm_filtered = median_tpm.loc[median_tpm.index.isin(expressed_gene_ids)]

smtsd_to_smts = sample_meta.drop_duplicates("SMTSD").set_index("SMTSD")["SMTS"].to_dict()
tissue_to_organ_map = {c: smtsd_to_smts.get(c, c) for c in median_tpm.columns}
_unmapped = sorted({c for c, o in tissue_to_organ_map.items() if o == c and c not in smtsd_to_smts.values()})
if _unmapped:
    raise KeyError(f"{len(_unmapped)} median_tpm columns did not map to SMTS, e.g. {_unmapped[:5]}")
print(f"Organs from median_tpm columns: {sorted(set(tissue_to_organ_map.values()))}")

non_testis_cols = [c for c in median_tpm.columns if tissue_to_organ_map[c] != "Testis"]

tau_all = compute_tau(median_tpm_filtered)
tau_nt = compute_tau(median_tpm_filtered[non_testis_cols])
tau_nt.name = "tau_no_testis"

tau_hybrid = (
    tau_all.rename("tau").reset_index().rename(columns={"index": "Name"})
    .merge(
        gencode_ids[["gene_id_v47", "gene_type_grouped"]],
        left_on="Name", right_on="gene_id_v47", how="inner",
    )
)
tau_hybrid = tau_hybrid[tau_hybrid["gene_type_grouped"].isin(organ_bar_biotypes)].dropna(subset=["tau"]).copy()
tau_hybrid["tau_category"] = tau_hybrid["tau"].apply(get_tau_category)
tau_hybrid = tau_hybrid.merge(
    tau_nt.reset_index().rename(columns={"index": "Name"}), on="Name", how="left"
)
tau_hybrid["tau_category_no_testis"] = tau_hybrid["tau_no_testis"].apply(get_tau_category)

tissue_specific = tau_hybrid[tau_hybrid["tau_category"] == "Tissue specific"].copy()
tpm_all = median_tpm.loc[tissue_specific["Name"]]
tpm_nt = median_tpm.loc[tissue_specific["Name"], non_testis_cols]

gene_max = pd.DataFrame({
    "max_tissue_all": tpm_all.idxmax(axis=1),
    "max_tissue_no_testis": tpm_nt.idxmax(axis=1),
    "gene_type_grouped": tissue_specific.set_index("Name")["gene_type_grouped"],
    "tau_category_no_testis": tissue_specific.set_index("Name")["tau_category_no_testis"],
})
gene_max["max_organ_all"] = gene_max["max_tissue_all"].map(lambda t: tissue_to_organ_map.get(t, t))
is_testis_peak = gene_max["max_organ_all"] == "Testis"
is_ts_no_testis = gene_max["tau_category_no_testis"] == "Tissue specific"
gene_max["ts_organ_batch"] = np.where(
    ~is_testis_peak,
    "non_testis_peak",
    np.where(is_ts_no_testis, "rescued_from_testis", "testis_only"),
)
gene_max["max_tissue"] = np.where(
    gene_max["ts_organ_batch"] == "rescued_from_testis",
    gene_max["max_tissue_no_testis"],
    gene_max["max_tissue_all"],
)
gene_max["max_organ"] = gene_max["max_tissue"].map(lambda t: tissue_to_organ_map.get(t, t))
gene_max["gene_type_grouped"] = pd.Categorical(
    gene_max["gene_type_grouped"], categories=organ_bar_biotypes, ordered=True
)

counts_before = (
    gene_max.groupby(["gene_type_grouped", "max_organ_all"], observed=False)
    .size()
    .rename("n_before")
    .reset_index()
    .rename(columns={"max_organ_all": "organ"})
)
counts_after = (
    gene_max.groupby(["gene_type_grouped", "max_organ"], observed=False)
    .size()
    .rename("n_after")
    .reset_index()
    .rename(columns={"max_organ": "organ"})
)
organ_counts = counts_before.merge(
    counts_after, on=["gene_type_grouped", "organ"], how="outer"
).fillna(0)
organ_counts["n_before"] = organ_counts["n_before"].astype(int)
organ_counts["n_after"] = organ_counts["n_after"].astype(int)

organ_totals_after = organ_counts.groupby("organ")["n_after"].sum().sort_values(ascending=False)
organs_sorted = organ_totals_after.index.tolist()
organ_label = {o: o for o in organs_sorted}

_batch_vc = gene_max["ts_organ_batch"].value_counts()
print(f"Tissue-specific genes: {len(gene_max):,}")
print(
    f"batches: non_testis_peak={int(_batch_vc.get('non_testis_peak', 0)):,}, "
    f"rescued_from_testis={int(_batch_vc.get('rescued_from_testis', 0)):,}, "
    f"testis_only={int(_batch_vc.get('testis_only', 0)):,}"
)
print(f"Panel d organs ({len(organs_sorted)}): {organs_sorted}")
print(
    "Testis before/after totals:",
    int(organ_counts.loc[organ_counts["organ"] == "Testis", "n_before"].sum()),
    "→",
    int(organ_counts.loc[organ_counts["organ"] == "Testis", "n_after"].sum()),
)


## Make figure


In [ ]:
fig = plt.figure(figsize=(fig_width, fig_height), dpi=fig_dpi)
gs = fig.add_gridspec(3, 1, height_ratios=[1, .75, 1.75], hspace=0.75)

hue_order = ["novel_lncRNA", "existing_lncRNA", "protein_coding", "pseudogene"]
ax_a_title = fig.add_subplot(gs[0, 0])
ax_a_title.set_axis_off()
gs_a = gs[0, 0].subgridspec(1, len(hue_order), wspace=0.35)
axes_a = [fig.add_subplot(gs_a[0, i]) for i in range(len(hue_order))]

for i, (gt, ax) in enumerate(zip(hue_order, axes_a)):
    gt_df = plot_df[plot_df["gene_type_grouped"] == gt]
    tissue_medians = []
    for tissue in loaded_tissues:
        vals = gt_df.loc[gt_df["Tissue"] == tissue, "Expression"].to_numpy(dtype=float)
        vals = vals[np.isfinite(vals) & (vals > 0)]
        if len(vals) < 5:
            continue
        tissue_medians.append(np.median(vals))
        sns.kdeplot(
            vals,
            ax=ax,
            color=tissue_palette.get(tissue, "0.5"),
            lw=0.8,
            alpha=0.75,
            log_scale=True,
            cut=0,
            bw_adjust=0.8,
        )
    median_median = float(np.median(tissue_medians)) if tissue_medians else np.nan
    if np.isfinite(median_median):
        ax.axvline(median_median, color="k", ls=":", lw=0.7, alpha=0.6)
        ax.annotate(
            f"{median_median:.2g}",
            xy=(median_median, 0.98),
            xycoords=("data", "axes fraction"),
            xytext=(4, 0),
            textcoords="offset points",
            va="top",
            ha="left",
            color="0.2",
        )
    ax.set_xscale("log")
    ax.set_xlim(1e-3, 1e5)
    ax.set_ylim(bottom=0)
    ax.set_ylabel(f"{gene_type_pretty[gt]}\ndensity")
    ax.set_yticks([])
    ax.set_xlabel("Median TPM")
    sns.despine(ax=ax)

ax_row = fig.add_subplot(gs[1, 0])
ax_row.set_axis_off()
inner = gs[1, 0].subgridspec(1, 5, wspace=0.85)
tau_gene_type_order = ["novel_lncRNA", "existing_lncRNA", "protein_coding", "pseudogene"][::-1]
category_colors = sns.color_palette("Greys", 3)

axes_b = [fig.add_subplot(inner[0, i]) for i in range(5)]
for i, (bin_label, ax_hist) in enumerate(zip(tau_strat["expr_bin"].cat.categories, axes_b)):
    bin_data = tau_strat[tau_strat["expr_bin"] == bin_label]
    for left, right, color in zip(category_edges[:-1], category_edges[1:], category_colors):
        ax_hist.axvspan(left, right, color=color, alpha=0.4, zorder=0, linewidth=0)
    for gt in tau_gene_type_order:
        gt_data = bin_data[bin_data["gene_type_grouped"] == gt]["tau"]
        if len(gt_data) > 0:
            sns.histplot(
                gt_data,
                label=gene_type_pretty.get(gt, gt) if i == 0 else None,
                color=gene_type_palette.get(gt),
                ax=ax_hist,
                element="step",
                lw=.8,
                stat="percent",
                cumulative=True,
                bins=500,
                binrange=(0, 1),
                fill=False,
                zorder=2,
            )
    ax_hist.set_xlim(0, 1)
    ax_hist.set_xticks([0, 0.5, 1])
    ax_hist.set_title(expr_bin_titles[bin_label], fontsize=6, pad=2)
    ax_hist.set_ylabel("cumulative %")
    ax_hist.set_xlabel("τ")
    sns.despine(ax=ax_hist)

gs_bottom = gs[2, 0].subgridspec(1, 2, width_ratios=[0.5, 1.15], wspace=0.35)
ax_c = fig.add_subplot(gs_bottom[0, 0])
ax_d = fig.add_subplot(gs_bottom[0, 1])

# --- c: genomic-context ORs ---
for i in range(len(lnc_cat_order)):
    if i % 2 == 0:
        ax_c.axhspan(i - 0.5, i + 0.5, color="#f5f5f5", zorder=0)

gt_offsets = {"novel_lncRNA": 0.15, "existing_lncRNA": -0.15}
for gt in odds_gene_types:
    gt_odds = odds_df[odds_df["gene_type_grouped"] == gt]
    color = gene_type_palette[gt]
    for i, cat in enumerate(lnc_cat_order):
        row = gt_odds[gt_odds["genomic_context"] == cat].iloc[0]
        y = i + gt_offsets[gt]
        or_log2 = np.log2(row.OR) if row.OR > 0 else np.nan
        lo = np.log2(row.CI_low) if row.CI_low > 0 else np.nan
        hi = np.log2(row.CI_high) if row.CI_high > 0 else np.nan
        ax_c.plot([lo, hi], [y, y], color=color, linewidth=0.8, zorder=1)
        is_ref = cat == reference_cat
        ax_c.scatter(
            or_log2, y, color=color, s=15, marker=("D" if is_ref else "o"),
            edgecolor="k", linewidths=0.35, zorder=3,
            label=gene_type_pretty[gt] if i == 0 else None,
        )
        if is_ref and gt == "novel_lncRNA":
            ax_c.text(or_log2 + 0.13, i, "Reference", va="center", ha="left", style="italic", fontsize=5)
        else:
            stars = sig_stars(row.pvalue)
            if stars and np.isfinite(or_log2):
                ax_c.text(or_log2, y + 0.1, stars, va="center", ha="right", fontsize=4)

ax_c.axvline(0.0, color="black", linestyle="--", linewidth=1, alpha=0.5, zorder=0)
ax_c.set_ylim(-0.5, len(lnc_cat_order) - 0.5)
ax_c.set_yticks(range(len(lnc_cat_order)))
ax_c.set_yticklabels([lnc_cat_pretty.get(c, c) for c in lnc_cat_order])
ax_c.tick_params(axis="y", length=0)
ax_c.set_xlabel("OR T.S. ~ expr. + genomic context")
ax_c.set_xticks([-2, -1, 0, 1])
ax_c.set_xticklabels(["1/4", "1/2", "1", "2"])
ax_c.legend(frameon=False, loc="lower center", fontsize=5, bbox_to_anchor=(0.5, 1), ncol=2)
sns.despine(ax=ax_c, left=True)

# --- d: peak-organ counts before/after Testis rescue ---
n_organs = len(organs_sorted)
n_gt = len(organ_bar_biotypes)
x_base = np.arange(n_organs)
biotype_offsets = (np.arange(n_gt) - (n_gt - 1) / 2.0) * 0.18

for i_gt, gt in enumerate(organ_bar_biotypes):
    color = gene_type_palette[gt]
    sub = organ_counts[organ_counts["gene_type_grouped"] == gt].set_index("organ")
    n_before = sub["n_before"].reindex(organs_sorted, fill_value=0).to_numpy(dtype=float)
    n_after = sub["n_after"].reindex(organs_sorted, fill_value=0).to_numpy(dtype=float)
    x = x_base + biotype_offsets[i_gt]

    for xi, yb, ya in zip(x, n_before, n_after):
        if yb == 0 or ya == 0:
            continue
        ax_d.plot([xi, xi], [yb, ya], color=color, lw=0.7, zorder=1, solid_capstyle="round")
    mask_before = n_before > 0
    mask_after = n_after > 0
    ax_d.scatter(
        x[mask_before], n_before[mask_before], s=4, facecolors="white", edgecolors=color, linewidths=0.6,
        zorder=3, label=None,
    )
    ax_d.scatter(
        x[mask_after], n_after[mask_after], s=4, facecolors=color, edgecolors=color, linewidths=0.4,
        zorder=3, label=gene_type_pretty[gt],
    )

ax_d.set_xticks(x_base)
ax_d.set_xticklabels(
    [organ_label.get(o, o) for o in organs_sorted],
    rotation=90, fontsize=4.5, ha="center",
)
ax_d.set_ylabel("Tissue-specific genes")
ax_d.set_xlabel("Peak organ")
ax_d.set_xlim(-0.6, n_organs - 0.4)
ax_d.set_yscale("log")
y_nonzero_vals = organ_counts[["n_before", "n_after"]].values.flatten()
y_nonzero_vals = y_nonzero_vals[y_nonzero_vals > 0]
if len(y_nonzero_vals) > 0:
    min_y = np.nanmin(y_nonzero_vals)
    ax_d.set_ylim(bottom=min_y * 0.85)
else:
    ax_d.set_ylim(bottom=1)
sns.despine(ax=ax_d)

legend_handles = [
    Line2D([0], [0], marker="o", color="0.4", markerfacecolor="white",
           markeredgecolor="0.4", markersize=4, lw=0, label="T.S. with Testis"),
    Line2D([0], [0], marker="o", color="0.4", markerfacecolor="0.4",
           markeredgecolor="0.4", markersize=4, lw=0, label="T.S. without Testis"),
] + [
    Line2D([0], [0], color=gene_type_palette[gt], lw=1.2, label=gene_type_pretty[gt])
    for gt in organ_bar_biotypes
]
ax_d.legend(
    handles=legend_handles, loc="upper right", frameon=False, fontsize=4.5,
    handlelength=1.2, labelspacing=0.25, ncol=1, bbox_to_anchor=(1, 1),
)

_inv = fig.transFigure.inverted()
# a/b/c share left-column x; d stays on its own axes
_x = _inv.transform(ax_a_title.transAxes.transform((-0.1, 0)))[0]
for ax, label in [(ax_a_title, "a"), (ax_row, "b"), (ax_c, "c"), (ax_d, "d")]:
    _y = _inv.transform(ax.transAxes.transform((0, 1.08)))[1]
    fig.text(
        _x if label in ("a", "b", "c") else _inv.transform(ax.transAxes.transform((-0.1, 0)))[0],
        _y, label, fontweight="bold", va="bottom", ha="right", fontsize=9,
    )
plt.show()

## Save figure


In [ ]:
fig_dir.mkdir(parents=True, exist_ok=True)
fig.savefig(fig_pdf, bbox_inches="tight")
fig.savefig(fig_png, bbox_inches="tight")
print(f"Wrote {fig_pdf}")
print(f"Wrote {fig_png}")

## Print stats


In [ ]:
# No numeric values from this figure are cited in the paper text or caption
# (qualitative support for Fig. 2 tissue-specificity analyses).
